<a href="https://colab.research.google.com/github/maureljean55/atelier-saint-honore/blob/main/notebooks/02_q1_q2_descriptif.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
clients = pd.read_csv("clients_propre.csv")
boutiques = pd.read_csv("boutiques_propre.csv")

print("clients :", clients.shape)
print("boutiques :", boutiques.shape)

clients : (700, 13)
boutiques : (24, 5)


In [2]:
clients["evolution_panier_pct"] = (clients["montant_prochain_achat_eur"] / clients["panier_moyen_eur"] - 1) * 100

clients[["client_id", "panier_moyen_eur", "montant_prochain_achat_eur", "evolution_panier_pct"]].head()

,client_id,panier_moyen_eur,montant_prochain_achat_eur,evolution_panier_pct
0,CT80059,1468.0,725.0,-50.613079
1,CT80578,1235.0,1479.0,19.757085
2,CT80256,1814.0,2307.0,27.177508
3,CT80098,1795.0,1820.0,1.392758
4,CT80362,2022.0,1201.0,-40.603363


In [3]:
print("Médiane de l'évolution :", round(clients["evolution_panier_pct"].median(), 1), "%")
print("Moyenne de l'évolution :", round(clients["evolution_panier_pct"].mean(), 1), "%")

part_en_baisse = (clients["evolution_panier_pct"] < 0).mean() * 100
print("Part des clients dont le panier baisse :", round(part_en_baisse, 1), "%")

Médiane de l'évolution : -23.1 %
Moyenne de l'évolution : -15.2 %
Part des clients dont le panier baisse : 76.9 %


In [4]:
boutiques["ville"] = boutiques["ville"].str.strip().str.lower()

pays_des_villes = {
    "paris": "France",
    "cannes": "France",
    "bordeaux": "France",
    "dubaï": "EAU",
    "shanghai": "Chine",
    "séoul": "Corée",
    "genève": "Suisse",
    "new york": "USA",
    "londres": "Royaume-Uni"
}

boutiques["pays_boutique"] = boutiques["ville"].map(pays_des_villes)

print("Villes sans pays :", boutiques["pays_boutique"].isna().sum())
boutiques[["ville", "pays_boutique"]].drop_duplicates()

Villes sans pays : 0


,ville,pays_boutique
0,dubaï,EAU
1,shanghai,Chine
2,cannes,France
4,genève,Suisse
5,séoul,Corée
6,paris,France
10,bordeaux,France
11,new york,USA
15,londres,Royaume-Uni


In [5]:
clients = clients.merge(
    boutiques[["boutique_id", "ville", "pays_boutique"]],
    left_on="boutique_principale",
    right_on="boutique_id",
    how="left"
)
clients = clients.drop(columns="boutique_id")

print(clients.shape)
clients[["client_id", "boutique_principale", "ville", "pays_boutique", "pays_residence"]].head()

(700, 16)


,client_id,boutique_principale,ville,pays_boutique,pays_residence
0,CT80059,BT15,séoul,Corée,usa
1,CT80578,BT28,cannes,France,france
2,CT80256,BT27,dubaï,EAU,france
3,CT80098,BT16,paris,France,italie
4,CT80362,BT31,dubaï,EAU,chine


In [6]:
clients["touriste"] = clients["pays_residence"].str.lower() != clients["pays_boutique"].str.lower()

clients["touriste"].value_counts()

,count
touriste,
True,549
False,151


In [7]:
clients.groupby("touriste")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
touriste,,,
False,-22.4,-14.5,151
True,-23.3,-15.3,549


In [8]:
clients.groupby("canal_prefere")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
canal_prefere,,,
boutique,-22.9,-16.9,404
en ligne,-23.8,-13.1,157
mixte,-23.3,-12.5,139


In [9]:
clients.groupby("carte_fidelite")["evolution_panier_pct"].agg(["median", "mean", "count"]).round(1)

,median,mean,count
carte_fidelite,,,
argent,-27.8,-22.7,242
aucune,-26.9,-20.7,336
or,9.6,23.1,101
